In [1]:
from pyspark.sql.functions import (
    col, trim, upper, when, current_timestamp,
    lit, to_date, datediff, round as spark_round
)
from pyspark.sql.types import DecimalType, IntegerType
from datetime import datetime

BRONZE_TABLE  = "Interac_Bronze.dbo.disputes"
SILVER_TABLE  = "silver_disputes"
SILVER_DB     = "Interac_Fabric_Workspace.Interac_Silver.dbo"
PIPELINE_NAME = "NB_06_Silver_Disputes"
BATCH_DATE    = datetime.now().strftime("%Y-%m-%d")

print(f"Silver Disputes Pipeline")
print(f"Started: {datetime.now()}")

StatementMeta(, 3de6da3d-4f69-4cbd-8430-4cb15aa94e07, 3, Finished, Available, Finished, False)

Silver Disputes Pipeline
Started: 2026-05-06 00:26:44.471352


In [2]:
df_bronze = spark.read.table(BRONZE_TABLE)
total_bronze = df_bronze.count()
print(f"Bronze rows read: {total_bronze:,}")

dq_results = {}
dq_results["null_dispute_id"] = df_bronze.filter(col("dispute_id").isNull()).count()
dq_results["null_cardholder_id"] = df_bronze.filter(col("cardholder_id").isNull()).count()
dq_results["null_disputed_amount"] = df_bronze.filter(col("disputed_amount_cad").isNull()).count()
dq_results["open_disputes"] = df_bronze.filter(col("status") == "OPEN").count()
dq_results["escalated_disputes"] = df_bronze.filter(col("is_escalated") == "Y").count()
dq_results["fraud_related"] = df_bronze.filter(col("is_fraud_related") == "Y").count()
dq_results["critical_priority"] = df_bronze.filter(col("priority") == "CRITICAL").count()
dq_results["missing_resolution"] = df_bronze.filter(
    (col("status").isin("RESOLVED_CARDHOLDER","RESOLVED_MERCHANT")) &
    col("resolved_date").isNull()).count()

print("\nDQ CHECK RESULTS:")
print("-" * 45)
for check, count_val in dq_results.items():
    status = "⚠ FLAGGED" if count_val > 0 else "✓ PASSED"
    print(f"{check:<35} {count_val:>6,}  {status}")

StatementMeta(, 3de6da3d-4f69-4cbd-8430-4cb15aa94e07, 4, Finished, Available, Finished, False)

Bronze rows read: 3,500

DQ CHECK RESULTS:
---------------------------------------------
null_dispute_id                          0  ✓ PASSED
null_cardholder_id                       0  ✓ PASSED
null_disputed_amount                     0  ✓ PASSED
open_disputes                          606  ⚠ FLAGGED
escalated_disputes                     115  ⚠ FLAGGED
fraud_related                        1,168  ⚠ FLAGGED
critical_priority                      240  ⚠ FLAGGED
missing_resolution                       0  ✓ PASSED


In [3]:
df_quarantine = df_bronze.filter(
    col("dispute_id").isNull() |
    col("cardholder_id").isNull() |
    col("disputed_amount_cad").isNull()
)
quarantine_count = df_quarantine.count()

if quarantine_count > 0:
    (df_quarantine
        .withColumn("_quarantine_reason", lit("NULL_PRIMARY_KEY_OR_AMOUNT"))
        .withColumn("_quarantined_at", current_timestamp())
        .write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_DB}.silver_disputes_quarantine"))
    print(f"Quarantined: {quarantine_count:,} records")

df_valid = df_bronze.filter(
    col("dispute_id").isNotNull() &
    col("cardholder_id").isNotNull() &
    col("disputed_amount_cad").isNotNull()
)

df_silver = (df_valid
    .withColumn("dispute_id",       trim(col("dispute_id")))
    .withColumn("cardholder_id",    trim(col("cardholder_id")))
    .withColumn("merchant_id",      trim(col("merchant_id")))
    .withColumn("dispute_reason",   upper(trim(col("dispute_reason"))))
    .withColumn("status",           upper(trim(col("status"))))
    .withColumn("priority",         upper(trim(col("priority"))))
    .withColumn("currency",         upper(trim(col("currency"))))
    .withColumn("is_fraud_related", upper(trim(col("is_fraud_related"))))
    .withColumn("is_escalated",     upper(trim(col("is_escalated"))))
    .withColumn("resolution_method",upper(trim(col("resolution_method"))))
    .withColumn("transaction_date",
        to_date(col("transaction_date"), "yyyy-MM-dd"))
    .withColumn("filed_date",
        to_date(col("filed_date"), "yyyy-MM-dd"))
    .withColumn("sla_deadline_date",
        to_date(col("sla_deadline_date"), "yyyy-MM-dd"))
    .withColumn("resolved_date",
        to_date(col("resolved_date"), "yyyy-MM-dd"))
    .withColumn("disputed_amount_cad",
        col("disputed_amount_cad").cast(DecimalType(18, 2)))
    .withColumn("resolution_amount_cad",
        when(col("resolution_amount_cad") == "", None)
        .otherwise(col("resolution_amount_cad"))
        .cast(DecimalType(18, 2)))
    .withColumn("days_to_resolve",
        when(col("resolved_date").isNotNull(),
            datediff(col("resolved_date"), col("filed_date")))
        .otherwise(None))
    .withColumn("is_sla_breached",
        when(
            col("resolved_date").isNotNull() &
            (col("resolved_date") > col("sla_deadline_date")), "Y")
        .when(col("resolved_date").isNull(), "PENDING")
        .otherwise("N"))
    .withColumn("recovery_rate_pct",
        when(
            col("resolution_amount_cad").isNotNull() &
            (col("disputed_amount_cad") > 0),
            spark_round(
                col("resolution_amount_cad") /
                col("disputed_amount_cad") * 100, 2))
        .otherwise(None))
    .withColumn("is_resolved",
        when(col("status").isin(
            "RESOLVED_CARDHOLDER","RESOLVED_MERCHANT","WITHDRAWN"),
            "Y").otherwise("N"))
    .withColumn("_silver_loaded_at", current_timestamp())
    .withColumn("_pipeline_name",    lit(PIPELINE_NAME))
    .withColumn("_batch_date",       lit(BATCH_DATE))
    .drop("_ingested_at", "_source_file", "_lakehouse")
)

print(f"Valid records: {df_silver.count():,}")

StatementMeta(, 3de6da3d-4f69-4cbd-8430-4cb15aa94e07, 5, Finished, Available, Finished, False)

Valid records: 3,500


In [4]:
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .saveAsTable(f"{SILVER_DB}.{SILVER_TABLE}"))

spark.sql(f"OPTIMIZE {SILVER_DB}.{SILVER_TABLE} ZORDER BY (filed_date, cardholder_id)")

final_count = spark.read.table(f"{SILVER_DB}.{SILVER_TABLE}").count()

print("\n" + "="*60)
print("SILVER DISPUTES SUMMARY")
print("="*60)
print(f"Bronze rows in    : {total_bronze:,}")
print(f"Quarantined       : {quarantine_count:,}")
print(f"Silver rows out   : {final_count:,}")
print(f"Pass rate         : {round(final_count/total_bronze*100, 2)}%")
print(f"Table             : {SILVER_DB}.{SILVER_TABLE}")
print(f"Completed at      : {datetime.now()}")
print("="*60)

StatementMeta(, 3de6da3d-4f69-4cbd-8430-4cb15aa94e07, 6, Finished, Available, Finished, True)


SILVER DISPUTES SUMMARY
Bronze rows in    : 3,500
Quarantined       : 0
Silver rows out   : 3,500
Pass rate         : 100.0%
Table             : Interac_Fabric_Workspace.Interac_Silver.dbo.silver_disputes
Completed at      : 2026-05-06 00:27:35.424149
